# Titanic Survival Prediction

## Data Preprocessing

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

import warnings
warnings.filterwarnings("ignore")

In [2]:
df_train = pd.read_csv("../data/raw/train.csv")
df_test = pd.read_csv("../data/raw/test.csv")

## Handle Missing Values

### Check Missing Values

In [3]:
print("Train Missing Values")
print(df_train.isnull().sum())

print("\nTest Missing Values")
print(df_test.isnull().sum())

Train Missing Values
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

Test Missing Values
PassengerId      0
Pclass           0
Name             0
Sex              0
Age             86
SibSp            0
Parch            0
Ticket           0
Fare             1
Cabin          327
Embarked         0
dtype: int64


### Fill Missing Values

In [4]:
# Age
df_train["Age"] = df_train["Age"].fillna(
    df_train["Age"].mean()
)

df_test["Age"] = df_test["Age"].fillna(
    df_test["Age"].mean()
)


# Fare
df_test["Fare"] = df_test["Fare"].fillna(
    df_test["Fare"].median()
)


# Embarked
df_train["Embarked"] = df_train["Embarked"].fillna(
    df_train["Embarked"].mode()[0]
)

## Feature Engineering

### Extract Title

In [5]:
df_train["Title"] = df_train["Name"].str.extract(
    ' ([A-Za-z]+)\.'
)

df_test["Title"] = df_test["Name"].str.extract(
    ' ([A-Za-z]+)\.'
)


rare_titles = [
    "Lady",
    "Countess",
    "Capt",
    "Col",
    "Don",
    "Dona",
    "Dr",
    "Major",
    "Rev",
    "Sir",
    "Jonkheer"
]


df_train["Title"] = df_train["Title"].replace(
    rare_titles,
    "Rare"
)

df_test["Title"] = df_test["Title"].replace(
    rare_titles,
    "Rare"
)

encoder = LabelEncoder()

df_train["Title"] = encoder.fit_transform(
    df_train["Title"]
)

df_test["Title"] = encoder.transform(
    df_test["Title"]
)

### Create FamilySize

In [6]:
df_train["FamilySize"] = (
    df_train["SibSp"] +
    df_train["Parch"] +
    1
)


df_test["FamilySize"] = (
    df_test["SibSp"] +
    df_test["Parch"] +
    1
)

### Create IsAlone

In [7]:
df_train["IsAlone"] = 0
df_test["IsAlone"] = 0


df_train.loc[
    df_train["FamilySize"] == 1,
    "IsAlone"
] = 1


df_test.loc[
    df_test["FamilySize"] == 1,
    "IsAlone"
] = 1

### Create AgeGroup

In [8]:
bins = [
    0,
    12,
    18,
    35,
    60,
    100
]


labels = [
    0,
    1,
    2,
    3,
    4
]


df_train["AgeGroup"] = pd.cut(
    df_train["Age"],
    bins=bins,
    labels=labels
)


df_test["AgeGroup"] = pd.cut(
    df_test["Age"],
    bins=bins,
    labels=labels
)

### Convert AgeGroup to Integer

In [9]:
df_train["AgeGroup"] = (
    df_train["AgeGroup"]
    .astype(int)
)


df_test["AgeGroup"] = (
    df_test["AgeGroup"]
    .astype(int)
)

### Remove Unused Columns

In [10]:
df_train = df_train.drop(
    columns=[
        "Name",
        "Ticket",
        "Cabin"
    ]
)


df_test = df_test.drop(
    columns=[
        "Name",
        "Ticket",
        "Cabin"
    ]
)

In [11]:
df_test

,PassengerId,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,Title,FamilySize,IsAlone,AgeGroup
0,892,3,male,34.50000,0,0,7.8292,Q,4,1,1,2
1,893,3,female,47.00000,1,0,7.0000,S,5,2,0,3
2,894,2,male,62.00000,0,0,9.6875,Q,4,1,1,4
3,895,3,male,27.00000,0,0,8.6625,S,4,1,1,2
4,896,3,female,22.00000,1,1,12.2875,S,5,3,0,2
...,...,...,...,...,...,...,...,...,...,...,...,...
413,1305,3,male,30.27259,0,0,8.0500,S,4,1,1,2
414,1306,1,female,39.00000,0,0,108.9000,C,7,1,1,3
415,1307,3,male,38.50000,0,0,7.2500,S,4,1,1,3
416,1308,3,male,30.27259,0,0,8.0500,S,4,1,1,2


In [12]:
df_train

,PassengerId,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,Title,FamilySize,IsAlone,AgeGroup
0,1,0,3,male,22.000000,1,0,7.2500,S,4,2,0,2
1,2,1,1,female,38.000000,1,0,71.2833,C,5,2,0,3
2,3,1,3,female,26.000000,0,0,7.9250,S,1,1,1,2
3,4,1,1,female,35.000000,1,0,53.1000,S,5,2,0,2
4,5,0,3,male,35.000000,0,0,8.0500,S,4,1,1,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,male,27.000000,0,0,13.0000,S,7,1,1,2
887,888,1,1,female,19.000000,0,0,30.0000,S,1,1,1,2
888,889,0,3,female,29.699118,1,2,23.4500,S,1,4,0,2
889,890,1,1,male,26.000000,0,0,30.0000,C,4,1,1,2


## Feature Transformation

### Log Transform Fare

In [13]:
df_train["Fare"] = np.log1p(
    df_train["Fare"]
)


df_test["Fare"] = np.log1p(
    df_test["Fare"]
)

df_train

In [14]:
df_test

,PassengerId,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,Title,FamilySize,IsAlone,AgeGroup
0,892,3,male,34.50000,0,0,2.178064,Q,4,1,1,2
1,893,3,female,47.00000,1,0,2.079442,S,5,2,0,3
2,894,2,male,62.00000,0,0,2.369075,Q,4,1,1,4
3,895,3,male,27.00000,0,0,2.268252,S,4,1,1,2
4,896,3,female,22.00000,1,1,2.586824,S,5,3,0,2
...,...,...,...,...,...,...,...,...,...,...,...,...
413,1305,3,male,30.27259,0,0,2.202765,S,4,1,1,2
414,1306,1,female,39.00000,0,0,4.699571,C,7,1,1,3
415,1307,3,male,38.50000,0,0,2.110213,S,4,1,1,3
416,1308,3,male,30.27259,0,0,2.202765,S,4,1,1,2


In [15]:
df_train

,PassengerId,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,Title,FamilySize,IsAlone,AgeGroup
0,1,0,3,male,22.000000,1,0,2.110213,S,4,2,0,2
1,2,1,1,female,38.000000,1,0,4.280593,C,5,2,0,3
2,3,1,3,female,26.000000,0,0,2.188856,S,1,1,1,2
3,4,1,1,female,35.000000,1,0,3.990834,S,5,2,0,2
4,5,0,3,male,35.000000,0,0,2.202765,S,4,1,1,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,male,27.000000,0,0,2.639057,S,7,1,1,2
887,888,1,1,female,19.000000,0,0,3.433987,S,1,1,1,2
888,889,0,3,female,29.699118,1,2,3.196630,S,1,4,0,2
889,890,1,1,male,26.000000,0,0,3.433987,C,4,1,1,2


## Encode Categorical Variables

## Encode Sex

In [16]:
df_train["Sex"] = df_train["Sex"].replace(
    {
        "male":1,
        "female":0
    }
)


df_test["Sex"] = df_test["Sex"].replace(
    {
        "male":1,
        "female":0
    }
)

### Encode Embarked

In [17]:
df_train["Embarked"] = df_train["Embarked"].replace(
    {
        "S":0,
        "C":1,
        "Q":2
    }
)


df_test["Embarked"] = df_test["Embarked"].replace(
    {
        "S":0,
        "C":1,
        "Q":2
    }
)

### Check

In [18]:
df_train.head()

,PassengerId,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,Title,FamilySize,IsAlone,AgeGroup
0,1,0,3,1,22.0,1,0,2.110213,0,4,2,0,2
1,2,1,1,0,38.0,1,0,4.280593,1,5,2,0,3
2,3,1,3,0,26.0,0,0,2.188856,0,1,1,1,2
3,4,1,1,0,35.0,1,0,3.990834,0,5,2,0,2
4,5,0,3,1,35.0,0,0,2.202765,0,4,1,1,2


In [19]:
df_test.head()

,PassengerId,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,Title,FamilySize,IsAlone,AgeGroup
0,892,3,1,34.5,0,0,2.178064,2,4,1,1,2
1,893,3,0,47.0,1,0,2.079442,0,5,2,0,3
2,894,2,1,62.0,0,0,2.369075,2,4,1,1,4
3,895,3,1,27.0,0,0,2.268252,0,4,1,1,2
4,896,3,0,22.0,1,1,2.586824,0,5,3,0,2


In [20]:
print(df_train.info())

print(df_test.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 13 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Sex          891 non-null    int64  
 4   Age          891 non-null    float64
 5   SibSp        891 non-null    int64  
 6   Parch        891 non-null    int64  
 7   Fare         891 non-null    float64
 8   Embarked     891 non-null    int64  
 9   Title        891 non-null    int64  
 10  FamilySize   891 non-null    int64  
 11  IsAlone      891 non-null    int64  
 12  AgeGroup     891 non-null    int64  
dtypes: float64(2), int64(11)
memory usage: 90.6 KB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  4

In [21]:
print("Train Missing:")
print(df_train.isnull().sum())


print("\nTest Missing:")
print(df_test.isnull().sum())

Train Missing:
PassengerId    0
Survived       0
Pclass         0
Sex            0
Age            0
SibSp          0
Parch          0
Fare           0
Embarked       0
Title          0
FamilySize     0
IsAlone        0
AgeGroup       0
dtype: int64

Test Missing:
PassengerId    0
Pclass         0
Sex            0
Age            0
SibSp          0
Parch          0
Fare           0
Embarked       0
Title          0
FamilySize     0
IsAlone        0
AgeGroup       0
dtype: int64


## Save Processed Dataset

In [22]:
df_train.to_csv(
    "../data/processed/train_processed.csv",
    index=False
)

df_test.to_csv(
    "../data/processed/test_processed.csv",
    index=False
)

print("Processed datasets saved successfully!")

Processed datasets saved successfully!
